# CPEN355 Cat Breed Classification - Workflow
Complete end-to-end pipeline from data download to inference.

In [ ]:
import sys, warnings, json
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import yaml

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}, PyTorch: {torch.__version__}")

In [ ]:
with open("configs/baseline.yaml") as f:
    config = yaml.safe_load(f)

dataset_id = config['data']['dataset_id']
raw_dir = Path(config['data']['raw_dir'])
processed_dir = Path(config['data']['processed_dir'])
selected_breeds = config['data']['selected_breeds']
image_size = config['data']['image_size']
batch_size = config['training']['batch_size']
num_epochs = config['training']['epochs']
learning_rate = config['training']['learning_rate']

print(f"Breeds: {', '.join(selected_breeds[:3])}... ({len(selected_breeds)} total)")

In [ ]:
def download_and_extract():
    if raw_dir.exists():
        print(f"✓ Dataset exists at {raw_dir}")
        return
    
    from kaggle.api.kaggle_api_extended import KaggleApi
    import shutil
    
    raw_parent = raw_dir.parent
    raw_parent.mkdir(parents=True, exist_ok=True)
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(dataset=dataset_id, path=str(raw_parent), unzip=True, quiet=False)
    
    nested = raw_dir / "images"
    if nested.exists() and nested.is_dir():
        for f in nested.iterdir():
            shutil.move(str(f), str(raw_dir))
        nested.rmdir()
    
    print(f"✓ Dataset ready")

download_and_extract()

In [ ]:
def build_metadata():
    image_paths, labels = [], []
    for breed_dir in raw_dir.iterdir():
        if not breed_dir.is_dir():
            continue
        for img_file in list(breed_dir.glob('*.jpg')) + list(breed_dir.glob('*.png')):
            image_paths.append(str(img_file))
            labels.append(breed_dir.name)
    
    df = pd.DataFrame({'image_path': image_paths, 'breed': labels})
    processed_dir.mkdir(parents=True, exist_ok=True)
    df = df[df['breed'].isin(selected_breeds)].reset_index(drop=True)
    df.to_csv(processed_dir / 'metadata.csv', index=False)
    print(f"✓ Metadata: {len(df)} images, {len(selected_breeds)} breeds")
    return df

metadata = build_metadata()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metadata['breed'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class Distribution', fontweight='bold')

for breed in sorted(selected_breeds):
    sample = metadata[metadata['breed'] == breed]['image_path'].iloc[0]
    try:
        img = Image.open(sample)
        axes[1].imshow(img)
        axes[1].set_title(f'Sample: {breed}', fontweight='bold')
        axes[1].axis('off')
        break
    except:
        pass

plt.tight_layout()
plt.show()

In [ ]:
def create_splits(df):
    tr, te = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['breed'])
    va, te = train_test_split(te, test_size=0.5, random_state=SEED, stratify=te['breed'])
    tr.to_csv(processed_dir / 'train.csv', index=False)
    va.to_csv(processed_dir / 'val.csv', index=False)
    te.to_csv(processed_dir / 'test.csv', index=False)
    print(f"✓ Splits: train={len(tr)}, val={len(va)}, test={len(te)}")
    return tr, va, te

train_df, val_df, test_df = create_splits(metadata)

In [ ]:
train_tr = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tr = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"✓ Transforms ready")

In [ ]:
breed_to_idx = {b: i for i, b in enumerate(sorted(selected_breeds))}
idx_to_breed = {v: k for k, v in breed_to_idx.items()}
with open(processed_dir / 'label_to_index.json', 'w') as f:
    json.dump(breed_to_idx, f)

class CatDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, breed_to_idx[row['breed']]

train_dl = DataLoader(CatDataset(train_df, train_tr), batch_size=batch_size, shuffle=True, pin_memory=True)
val_dl = DataLoader(CatDataset(val_df, eval_tr), batch_size=batch_size, shuffle=False, pin_memory=True)
test_dl = DataLoader(CatDataset(test_df, eval_tr), batch_size=batch_size, shuffle=False, pin_memory=True)
print(f"✓ DataLoaders: train={len(train_dl)}, val={len(val_dl)}, test={len(test_dl)} batches")

In [ ]:
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(breed_to_idx))
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, num_epochs // 3), gamma=0.1)
ckpt_dir = Path('outputs/checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ ResNet50: {total_params:,} parameters")

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(num_epochs):
    model.train()
    tr_loss, tr_cor, tr_tot = 0, 0, 0
    for img, lbl in tqdm(train_dl, desc=f"Epoch {epoch+1} [Train]", leave=False):
        img, lbl = img.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(model(img), lbl)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * img.size(0)
        tr_cor += (model(img).argmax(1) == lbl).sum().item()
        tr_tot += lbl.size(0)
    
    model.eval()
    va_loss, va_cor, va_tot = 0, 0, 0
    with torch.no_grad():
        for img, lbl in tqdm(val_dl, desc=f"Epoch {epoch+1} [Val]", leave=False):
            img, lbl = img.to(device), lbl.to(device)
            out = model(img)
            loss = criterion(out, lbl)
            va_loss += loss.item() * img.size(0)
            va_cor += (out.argmax(1) == lbl).sum().item()
            va_tot += lbl.size(0)
    
    tr_loss, tr_acc = tr_loss/tr_tot, tr_cor/tr_tot
    va_loss, va_acc = va_loss/va_tot, va_cor/va_tot
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    
    print(f"Epoch {epoch+1}: Loss {tr_loss:.4f}/{va_loss:.4f}, Acc {tr_acc:.4f}/{va_acc:.4f}")
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), ckpt_dir / 'best.pt')
    torch.save(model.state_dict(), ckpt_dir / 'last.pt')
    scheduler.step()

print(f"✓ Training done. Best: {best_val_acc:.4f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history['train_loss'], label='Train', marker='o')
a1.plot(history['val_loss'], label='Val', marker='s')
a1.set_title('Loss', fontweight='bold')
a1.legend()
a1.grid(True, alpha=0.3)

a2.plot(history['train_acc'], label='Train', marker='o')
a2.plot(history['val_acc'], label='Val', marker='s')
a2.set_title('Accuracy', fontweight='bold')
a2.legend()
a2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(ckpt_dir / 'best.pt', map_location=device))
model.eval()
va_pred, va_true = [], []
with torch.no_grad():
    for img, lbl in tqdm(val_dl, desc="Val", leave=False):
        img = img.to(device)
        va_pred.extend(model(img).argmax(1).cpu().numpy())
        va_true.extend(lbl.numpy())

print("Validation:")
print(f"  Accuracy: {accuracy_score(va_true, va_pred):.4f}")
print("\n" + classification_report(va_true, va_pred, target_names=[idx_to_breed[i] for i in range(len(breed_to_idx))]))

In [ ]:
te_pred, te_true = [], []
with torch.no_grad():
    for img, lbl in tqdm(test_dl, desc="Test", leave=False):
        img = img.to(device)
        te_pred.extend(model(img).argmax(1).cpu().numpy())
        te_true.extend(lbl.numpy())

te_acc = accuracy_score(te_true, te_pred)
print(f"Test Accuracy: {te_acc:.4f}")

cm = confusion_matrix(te_true, te_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[idx_to_breed[i][:8] for i in range(len(breed_to_idx))],
            yticklabels=[idx_to_breed[i][:8] for i in range(len(breed_to_idx))], ax=ax)
ax.set_title('Confusion Matrix', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Path('outputs/metrics').mkdir(parents=True, exist_ok=True)
np.save('outputs/metrics/confusion_matrix.npy', cm)

In [ ]:
def predict(img_path: str, top_k: int = 3):
    model.eval()
    img = Image.open(img_path).convert('RGB')
    tensor = eval_tr(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top_p, top_i = torch.topk(probs, top_k)
    return [{'breed': idx_to_breed[i.item()], 'prob': p.item()} for p, i in zip(top_p, top_i)]

preds = predict(test_df.iloc[0]['image_path'], top_k=3)
for i, p in enumerate(preds, 1):
    print(f"{i}. {p['breed']}: {p['prob']:.2%}")

img = Image.open(test_df.iloc[0]['image_path'])
plt.imshow(img)
plt.axis('off')
plt.title('\n'.join([f"{p['breed']}: {p['prob']:.1%}" for p in preds]), fontweight='bold')
plt.tight_layout()
plt.show()